# 토픽 분류 데이터셋 생성

## 목표
4개국(`러시아_Wildberries`, `미국_ULTA`, `일본_라쿠텐`, `한국_올리브영`) `_리뷰_형태소분석.xlsx` 파일의
`리뷰_형태소` 컬럼을 속성 키워드 사전과 매칭해서, 리뷰 1건 = 1행인 토픽 분류 데이터셋을 만든다.

## 산출물 2개
1. **토픽분류_데이터셋.xlsx** — `리뷰_번호`, `국가`, `제품_순위`, `별점`, `Y`, 속성 9개(`보습`,`향`,`발림성`,`자극`,`용기`,`배송`,`기능`, `가격`, `브랜드 충성도`)
   binary 컬럼만 포함. 추후 모델 학습을 위한 전처리용 데이터셋 생성.
2. **해석용_참고파일.xlsx** — `리뷰_번호`로 위 파일과 연결되는 `리뷰_원문`·`리뷰_불용어처리`·`리뷰_형태소`.
   모델에는 넣지 않고, 나중에 "이 리뷰가 왜 이 속성으로 분류됐는지" 확인할 때만 사용.

## 규칙
- **`리뷰_번호`**: `국가코드_P제품순위_리뷰번호` 형태로 새로 생성   
 (예: `US_P01_0001`).
  원본 파일의 `리뷰_번호`가 국가 전체에서 유일하지 않고 **제품_순위마다 1부터 다시 시작**하는 경우가 있어서
  (러시아·미국·한국 파일이 그렇고, 일본만 이미 전체 고유번호), 제품_순위를 포함시켜 국가 내에서 완전히
  유일한 ID가 되도록 함.
- **`Y`**: 별점 4·5 → 1, 1·2 → 0. 이전 단계 파일에 이미 계산되어 있어 그대로 사용(3점 리뷰는 이미 제외됨).
- **속성 9개**: 이전에 정리한 속성별 키워드 사전(`보습`,`향`,`발림성`,`자극`,`용기`,`배송`, `기능`, `가격`, `브랜드 충성도`)을 기준으로,
  키워드가 `리뷰_형태소`에 있으면 1, 없으면 0.
  - `리뷰_형태소`는 공백으로 구분된 유니그램(단어 1개)이므로, 키워드가 한 단어면 유니그램과, 두 단어(예: `효과 좋다`)면
    인접한 두 토큰(바이그램)과 **정확히 일치**하는지 확인. 단순 부분 문자열 비교를 하지 않는 이유:
    예를 들어 `번`(재구매 키워드)이 `번들거리다`(발림성) 안의 글자와 우연히 겹쳐서 잘못 매칭되는 걸 막기 위함.
  - 키워드 목록에 `추천하지_않다`처럼 `_`로 합쳐진 것과 `추천하지 않다`처럼 띄어쓴 것이 섞여 있어서,
    비교 전에 `_`을 공백으로 통일해 같은 표현으로 인식하도록 함.
- 리뷰를 여러 조각으로 쪼개지 않고, 리뷰 1건을 그대로 1행으로 유지.

In [1]:
!pip install -q openpyxl

In [2]:
import pandas as pd
from google.colab import files

# 4개국 '_리뷰_형태소분석.xlsx' 파일을 한 번에 선택하여 업로드
uploaded = files.upload()
print(list(uploaded.keys()))

Saving 러시아_Wildberries_리뷰_형태소분석.xlsx to 러시아_Wildberries_리뷰_형태소분석.xlsx
Saving 미국_ULTA_리뷰_형태소분석.xlsx to 미국_ULTA_리뷰_형태소분석.xlsx
Saving 일본_라쿠텐_리뷰_형태소분석.xlsx to 일본_라쿠텐_리뷰_형태소분석.xlsx
Saving 한국_올리브영_리뷰_형태소분석.xlsx to 한국_올리브영_리뷰_형태소분석.xlsx
['러시아_Wildberries_리뷰_형태소분석.xlsx', '미국_ULTA_리뷰_형태소분석.xlsx', '일본_라쿠텐_리뷰_형태소분석.xlsx', '한국_올리브영_리뷰_형태소분석.xlsx']


## 1. 국가 자동 인식 & 데이터 로드

파일명에 아래 키워드가 포함되어 있으면 해당 국가로 인식. `code`는 `리뷰_번호`에 쓸 국가 코드,
`name`은 최종 데이터셋의 `국가` 컬럼에 쓸 표시 이름.

In [3]:
COUNTRY_INFO = {
    '한국_올리브영': {'code': 'KR', 'name': '한국', 'keywords': ['올리브영', '한국']},
    '미국_ULTA': {'code': 'US', 'name': '미국', 'keywords': ['ULTA', '미국']},
    '일본_라쿠텐': {'code': 'JP', 'name': '일본', 'keywords': ['라쿠텐', '일본']},
    '러시아_Wildberries': {'code': 'RU', 'name': '러시아', 'keywords': ['Wildberries', '러시아']},
}


def match_country(filename: str):
    for key, info in COUNTRY_INFO.items():
        if any(kw in filename for kw in info['keywords']):
            return key
    return None


country_dfs = {}
for filename in uploaded.keys():
    key = match_country(filename)
    if key is None:
        print(f'[경고] 국가를 자동으로 인식하지 못했습니다: {filename} (건너뜀)')
        continue
    country_dfs[key] = pd.read_excel(filename)
    print(f'{key:20s} <- {filename}  {country_dfs[key].shape}')

assert len(country_dfs) > 0, '인식된 국가 데이터가 없습니다. COUNTRY_INFO를 확인하세요.'

러시아_Wildberries      <- 러시아_Wildberries_리뷰_형태소분석.xlsx  (1842, 9)
미국_ULTA              <- 미국_ULTA_리뷰_형태소분석.xlsx  (1907, 9)
일본_라쿠텐               <- 일본_라쿠텐_리뷰_형태소분석.xlsx  (1542, 9)
한국_올리브영              <- 한국_올리브영_리뷰_형태소분석.xlsx  (1939, 9)


## 2. 속성별 키워드 사전

- 9개 속성: 보습, 향, 발림성, 자극, 용기, 배송, 기능, 가격, 브랜드 충성도.

In [ ]:
## 속성사전 -> 딕셔너리 변환용 정규식
# 촉촉, 쫀쫀 -> '촉촉', '쫀쫀'

import re
text = '브랜드, 단골, 회사 신뢰, 시리즈, 라인 제품, 이번, 번, 번 주문, 번 구매, 여러 번, 처음, 구매 적 있다, 항상, 정기 구매, 크림 사용 보다, 마음 들다 주문, 오랫동안, 년 동안, 사용 오다, 년 사용 있다,'

print(re.sub(r'([^,]+)', lambda m: f"'{m.group(0).strip()}'", text).replace("','", "', '"))

'브랜드', '단골', '회사 신뢰', '시리즈', '라인 제품', '이번', '번', '번 주문', '번 구매', '여러 번', '처음', '구매 적 있다', '항상', '정기 구매', '크림 사용 보다', '마음 들다 주문', '오랫동안', '년 동안', '사용 오다', '년 사용 있다',


In [6]:
ATTRIBUTE_KEYWORDS = {
    '보습': [
        '촉촉', '쫀쫀', '보습', '보습 뛰어나다', '보습 되다', '보습제', '촉촉하다', '수분', '윤기', '크리미', '촉촉 유지', '촉촉 지다',
        '건조 피부', '윤기 나다', '피부 촉촉', '각질', '건조 만들다', '보습 효과', '거칠다', '건조', '각질 일어나다', '건조 지다', '기름',
        '기름지다', '기름지다 하다', '때 밀리다', '뜨다', '영양', '부드럽다 지다', '부드럽다 하다 지다', '리치', '산뜻', '피부 건조',
        '부드럽다 하다', '부드럽다 만들다', '피부 부드럽다', '피부 매끄럽다 만들다', '빠르다', '갈라지다 줄어들다', '매끄럽다 지다', '보습 확실',
        '피부 부드럽다 만들다', '피부 매끄럽다 지다', '매끈 지다', '보습 좋다', '피부 실크 부드럽다 지다', '피부 부드럽다 매끄럽다 만들다',
        '실크 부드럽다 지다', '벨벳 부드럽다', '부드럽다 가꾸다 주다', '모이스처라이징', '보습 부족', '건조 느낌', '보습 느끼지_못하다',
        '보습 좋지_않다', '보습 효과 좋지_않다', '손 부드럽다 지다', '보습 효과_없다', '갈라지다', '피부 하얗다 뜨다', '보습 탁월', '보습 안_되다',
        '갈라지', '심하다 건조', '갈라지다 발뒤꿈치', '보습 탁월하다', '촉촉하지_않다', '보습 떨어지다', '건조증', '건성 피부', '건조하지_않다',
        '오일 효과', '보습 효과 있다', '보습 효과 뛰어나다', '푸석', '하루 종일 촉촉하다', '손 트다', '보습 약하다', '피부 땅기다', '건기',
        '건조 만들지_않다', '건조 완화', '수분 공급', '당기다', '보습력', '유분', '영양 공급', '피부 조밀 지다', '기름지지_않다', '피부 매끄럽다',
        '매끄럽다 하다', '매끄럽다', '세안 후 매끄럽다', '서리 보호 주다', '숨 쉬다', '피부 벗겨지지_않다', '피부 바람 반응하지_않다', '건성',
        '가을', '겨울', '겨울철', '찬바람', '환절기', '땡기다', '습기 영향 받지_않다', '습기', '바람 보호 주다',
    ],
    '향': [
        '냄새', '냄새 나다', '냄새 심하다', '냄새 없다', '향', '향 없다', '향 좋다', '효과 향', '향 은은', '향 나다', '은은', '왁스 나다',
        '향 강하다', '화학', '불쾌', '화학 냄새', '왁스 향', '왁스 냄새', '체리 향', '냄새 지독', '퀴퀴', '고급 향', '코코넛 향', '천연 향',
        '냄새 민감', '향기', '향 달라지다', '향 처음 강하다', '향 상쾌', '향 민감', '처음 향 거슬리다', '처음 향 강하다', '처음 냄새 싫다',
        '티 트리 향', '상쾌', '아로마', '향 마음 편안', '향 강하지_않다', '처음 향 안_맞다', '라벤더', '향 거슬리다', '냄새 강하다',
        '냄새 고약', '뚜렷하지_않다', '무 향', '냄새 불쾌', '향 바뀌다', '고무 냄새', '로즈 향', '특유 냄새', '허브', '약 냄새', '진하다',
        '향 과하지_않다', '악취', '향수', '향 자극 않다',
    ],
    '발림성': [
        '질감', '질감 좋다', '제형', '감촉', '기분 좋다 질감', '가볍다', '산뜻', '끈적이지_않다', '매끄럽다 바르다', '끈적이다', '번들거리지_않다',
        '흡수', '흡수 빠르다', '흡수 되다', '흡수력 좋다', '흡수 않다', '발리다', '가볍다 산뜻', '번들거리다', '끈적거리다', '바르다 있다',
        '바르다 쉽다', '묽다', '묽지_않다', '오일', '겉돌다', '흡수 안_되다', '알갱이', '액체', '무겁다', '무겁다 느낌', '알갱이 남다', '가볍다 바르다',
        '무겁지_않다', '밀리다', '쓰다 좋다', '바르다', '진하다 크림', '시원 느낌', '사용 좋다', '피부 시원', '발림', '편안 사용', '농도 진하다',
        '되직', '되직 발리다', '부드럽다 하다', '발림 좋다', '촉감 좋다', '피부 밀착', '감촉 좋다', '되직하지_않다', '스며들다', '크리미 질감',
        '흡수력', '뭉치다', '펴다 발리지_않다', '바르다 불편', '바르다 편하다', '기분 좋다', '버터 부드럽다', '점도 높다', '펴다 바르다',
        '닿다 느낌 좋다', '스며들지_않다', '질감 거칠다', '고르지_않다', '사용 부드럽다', '피부 흡수 마음 들다', '얇다', '편하다', '묵직', '크리미',
        '피부 흡수 놀라다', '질감 가볍다', '두껍다', '뻑뻑하다', '매트', '밀착', '안_펴다', '안_발리다', '질감 부드럽다', '텍스처', '뭉치지_않다',
        '마무리 감 좋다', '부드럽지_않다', '들뜨지_않다', '농도', '밀림', '크림 질 좋지_않다', '하얗다 액체 같다', '느낌 들지_않다', '잔여 감 남다',
        '오일리하지_않다', '크림 같다 느낌 들지_않다', '시술 과정 편안', '피부 편안 느낌 유지',
    ],
    '자극': [
        '피부 맞다', '피부_안맞다', '피부 안_맞다', '안_맞다', '진정', '안심', '편하다', '진정 주다', '자극 없다', '순하다', '민감',
        '붉다', '사라지다', '아쉽다', '붓다', '자극', '붉어지다', '적합', '맞지_않다', '가렵다', '피부 적합하지_않다', '트러블 생기다', '피부과',
        '유발', '따끔거리다', '상처', '여드름', '안정', '장벽', '피부 가렵다', '알레르기', '뾰루지', '발진', '새빨개지다', '빨갛다', '여드름 나다',
        '좁쌀', '생기다', '나다', '걸리다', '벗겨지다', '자극 주다', '좁쌀 여드름', '발생', '발진 생기다', '반응 나타나다', '악화', '피부과 가다',
        '가려움증', '올라오다', '뒤집어지다', '피부 맞지_않다', '따갑다', '간지럽다', '여드름 올라오다', '예민', '트러블 나다',
        '시리다', '눈 시리다', '자극 순하다', '붉다 지다 있다', '피부염', '일어나다', '낫다', '순하다 좋다', '화농', '막히다',
        '트러블 생기지_않다', '트러블 유발', '트러블 없다', '눈 자극', '모공 막지_않다', '피부 타입 맞다', '트러블 사라지다', '줄다', '화끈거리다',
        '화상 입다', '자극 완화', '자극 않다', '트러블 유발하지_않다', '자극 받다', '도드라지다 하다', '문제 피부', '습진', '눈 따갑지_않다',
        '피부 타다 듯 느낌', '트러블 걱정 없다', '화학 화상 입히다', '트러블', '트러블 일으키지_않다', '트러블 깨끗', '모공 막다', '트러블 개선',
        '따갑지_않다', '피부 뜨겁다', '부풀다 오르다', '새빨갛다', '화상 부위', '트러블 줄다', '발생량', '민감 피부', '가라앉다', '여드름 심하다',
        '아토피 심하다', '여드름 있다', '관자놀이 턱 나다', '사용 후 여드름 사라지다', '볼 붉다', '여드름 고민', '아토피 부위', '사용 후 자극 없다',
        '여드름 도움', '피부 예민', '여드름 줄어들다', '턱 여드름', '여드름 줄다', '여름철 예민', '등드름', '성인 여드름', '화이트헤드', '홍조',
        '여드름 사라지다', '다음 날 진정', '피부 자극 나아지다', '여드름 없다', '여드름 효과 없다', '빨개지다', '부어오르다', '여드름 나아지지_않다',
        '여드름 급증', '지루성 피부염', '얼굴 따갑다', '눈 따갑다', '가렵지_않다', '생기지_않다', '피부 안정', '번지다', '자극 줄다',
        '자극 유발하지_않다', '피부 벗겨지다', '알레르기 반응 없다', '붉어지다 없다', '모공 넓다 지다', '뒤집다 지다', '필름 붙다 것 당기다',
        '숨 쉬다 힘들다', '피부 엉망', '편안 지다', '피부 약하다', '튼튼', '장벽 무너지다', '두드러기', '모낭', '예민하다', '회복 되다', '불거지다',
        '눈 시렵다'
    ],
    '용기': [
        '리필', '사용 편리', '색상', '롤러', '제품 손상', '내용물', '작동하지_않다', '디스펜서', '비다', '상자', '병', '분사기', '튜브', '노즐',
        '사진', '리뉴얼', '가짜', '뚜껑', '절반', '포장', '뚜껑 열다', '용기', '펌프', '고장', '많다', '설명 다르다', '열리다 있다', '열리다',
        '박스', '사이즈', '작다 사이즈', '크다 사이즈', '여행 사이즈', '용량 적다', '스패츌라', '디자인', '디자인 훌륭', '세련', '패키지', '용량',
        '용량 넉넉', '용량 좋다', '소량', '보호 필름 없다', '누르다', '휴대 좋다', '내용물 없다', '꺼내다 쓰다', '이상 안_나오다', '용기 작다',
        '사용 충분', '사용 쉽다', '유통기한', '유통 기한', '구성', '상태 양호', '소비량 적다', '짜내기 힘들다', '봉인 스티커', '미니사이즈',
        '샘플 사이즈', '뚜껑 안_열리다', '뚜껑 쉽다', '묻다 나오다', '안쪽 뚜껑', '통', '사용 편안', '간편', '구멍', '유리병', '용기 편리',
        '내용물 훌륭', '짜내다', '스프레이', '사용 편하다', '튜브 편리', '세트', '케이스', '용량 늘리다', '미니 사이즈', '밀봉 안_되다', '색깔 바래다',
        '구멍 생기다', '원터치', '내용물 안_나오다', '용기 윗부분', '용량 표시', '불편', '용기 넘어지다', '상품 들다 있다', '휴대 편리', '샘플',
        '필름 코팅', '버전 출시 일 년', '정품', '보관', '입구', '크다 용량', '양 적다', '쓰다 편하다'
    ],
    '배송': [
        '배송 빠르다', '빠르다 배송', '도착 빠르다', '도착', '일 빠르다', '배송', '도착하지_않다', '상태 도착', '구겨지다', '찌그러지다 채 도착',
        '주문 상품 받다 못', '제시간', '제시간 도착', '파손', '주문 다르다 상품', '잘못 상품 보내다', '수거 거부', '제품 누락', '느리다', '지나다',
        '주문 후 도착', '오다 실수', '배송 중 변질', '배송 시작 안_되다', '주문 못_받다', '주문 상품', '내용물 새다', '내용물 분리', '전화 연결 안_되다',
        '품목 누락', '누락', '손상', '손상 도착', '상자 밀봉', '찌그러지다', '개봉', '개봉 상태', '포장 되다', '3중 포장', '비다 있다', '뜯다 지다 있다',
        '안전 받다', '주문하지_않다 제품', '택배', '상자', '테이프', '찢기다', '찢어지다', '벌어지다', '검수', '배달', '예정일', '엉뚱',
        ],
    '기능': [
        '효과', '효과 느끼다', '효과_없다', '효과 있다', '효과 좋다', '사용 좋다', '변화 없다', '사용 효과 없다', '효과 나타나다',
        '탄력', '주름', '개선', '도움', '반응 없다', '효과 뛰어나다', '피부 좋다', '생기', '생기 있다', '효과 없다', '효과 모르다', '변하다',
        '효과 못_느끼다', '효과 못_보다', '품질 떨어지다', '도움 되지_않다', '재생', '피부 회복', '차이 느끼다', '잔주름 완화', '잔주름 개선',
        '효과 보다', '효과 우수', '리프팅 효과', '생기 더하다', '생기 불어넣다', '안티에이징', '스킨 케어', '자외선 차단', '잔주름 옅다 지다',
        '노화 방지', '피부 젊다', '생기 유지', '메이크업 전 사용 완벽', '흉터 옅다 지다', '피부 환하다', '기미 옅다', '피부 톤 밝히다', '깨끗',
        '메이크업 베이스', '좋다 결과', '결과 좋다', '역할', '역할 해내다', '보호막 형성', '잔여물 없다', '잔주름', '시간 단축', '회복 주다',
        '피부 보호', '빠르다 회복', '세척', '생기 되찾다', '피부 살아나다', '피부 살리다 주다', '환하다 지다', '생기 넘치다', '피부 어리다 보이다 하다',
        '피부 맑다 지다', '피부 톤 밝다 균일 지다', '피부 어리다 보이다', '여드름 없애다 주다', '깨끗 하다 주다', '잔주름 사라지다', '피부 톤 개선',
        '피부 톤 고르다', '항 노화', '피부 톤 균일 지다', '안티 에이징', '습진 없애다 주다', '피부 결 눈 띄다 좋다 지다', '빠르다 낫다', '햇볕 타다',
        '검다 자국 사라지다', '치료 효과', '품질 안_좋다', '효과 떨어지다', '피부 달라지다', '노폐물 제거 효과', '전신 쓰다', '환하다', '피부 결 좋다',
        '피부 깨끗', '모공 작아지다', '즉각 효과', '피부 상태 좋다', '화잘먹', '메이크업 밀착', '파운데이션 밀착', '메이크업 전', '제품 효과 있다',
        '메이크업 전 사용', '사용 후 눈 띄다 좋다', '나아지지_않다', '뚜렷 변화 못_느끼다', '주름 없다', '피부 탄력 있다', '이마 주름 옅다',
        '다크 서클 효과 좋다', '팔자 주름 효과 못_보다', '목 주름 사라지다', '잔주름 옅다', '얼굴 주름 옅다', '화장 먹다', '목 피부', '목 주름 옅다',
        '주름 사라지다', '주름 옅다', '피부 놀랍다', '하얘지다', '피부 눈 띄다', '하얗다', '화장 되다', '피부 조금 좋다', '메이크업 뭉치지_않다',
        '눈 밑 처지다 놀랍다', '광채', '윤기 있다', '주름 개선 효과', '효과 말씀 드리다', '개선 효과 느끼다', '피부 변화 느끼다', '피부 탄력',
        '주름 개선', '피부 톤 밝다', '탄력 유지', '눈 밑 주름 옅다', '성능 훌륭 것 같다', '목 주름', '세안 후 매끈', '잔주름 없다', '차이 못_느끼다',
        '피부 하얗다', '개선 효과 못_보다', '효과 느끼지_못하다', '미백', '자외선', '이마 주름 깊이 줄어들다', '피부 결', '피부 톤', '톤 밝다',
        '노화 방지 도움', '기미', '피부 맑다', '얼굴 환하다', '프라이머 대신', '피부 활력', '리프팅', '아이 크림 대신', '피부 상태 눈 띄다',
        '비타민 유도체', '기미 개선', '잡티', '레이저 시술', '레이저 급', '비타민 세럼 효과', '붓 줄이다 주다', '붓다 줄이다 주다', '결과 마음 들다',
        '성분', '잔주름 감소', '다크 서클 개선', '메이크업 안_먹다', '파운데이션 뭉치다', '품질', '품질 좋다', '품질 좋지_않다',
        '품질 낮다', '뛰어나다 품질', '품질 우수', '품질 훌륭', '네일', '붓지_않다', '결과 없다', '요소 포함', '변경 사항 없다',
        '바뀌다 수 있다', '피부 속 빛나다 것 화사 보이다', '피부 정돈 보이다', '피부 온도 변화 강하다 지다', '피부 깔끔 보이다',
        '용도 추천', '화 먹다', '베이스', '메이크업 먹다', '광 살아나다', '피부 광', '질 좋지_않다', '시간 절약', '불분명하다',
        '크림 덕분',
    ],
  '가격': [
      '가격', '가성비', '비싸다', '저렴', '가치', '할인', '쿠폰', '행사', '저렴하다', '추가 결제', '가격 좋다', '싸다', '반품 비 받다', '기획',
      '증정', '포카', '샘플', '사은품', '세트', '이벤트',
  ],
  '브랜드 충성도': [
      '브랜드', '단골', '회사 신뢰', '시리즈', '라인 제품', '이번', '번', '번 주문', '번 구매', '여러 번', '구매 적 있다', '항상',
      '정기 구매', '크림 사용 보다', '마음 들다 주문', '오랫동안', '년 동안', '사용 오다', '년 사용 있다', '구매 쓰다 보다'
  ]
}

ATTRIBUTE_COLUMNS = list(ATTRIBUTE_KEYWORDS.keys())
print('속성 컬럼:', ATTRIBUTE_COLUMNS)

속성 컬럼: ['보습', '향', '발림성', '자극', '용기', '배송', '기능', '가격', '브랜드 충성도']


## 3. 속성 매칭 함수

`리뷰_형태소`를 토픽 사전의 n개의 gram 집합(**키워드 최대 길이까지 동적으로 생성되도록 함**)으로 만든 뒤, 키워드가 그 집합에
**정확히** 포함되는지로 판정.

In [7]:
import re


def normalize(text: str) -> str:
    return re.sub(r'\s+', ' ', text.replace('_', ' ')).strip()


ATTRIBUTE_KEYWORDS_NORM = {
    attr: [normalize(k) for k in kws] for attr, kws in ATTRIBUTE_KEYWORDS.items()
}

# 키워드 중 가장 긴 것이 몇 단어인지에 맞춰 n-gram 길이를 동적으로 결정
MAX_GRAM_N = max(len(kw.split()) for kws in ATTRIBUTE_KEYWORDS_NORM.values() for kw in kws)


def build_gram_set(review_형태소: str, max_n: int = MAX_GRAM_N) -> set:
    tokens = normalize(review_형태소).split()
    grams = set()
    for n in range(1, max_n + 1):
        grams.update(' '.join(tokens[i:i + n]) for i in range(len(tokens) - n + 1))
    return grams


def match_attributes(review_형태소) -> dict:
    text = review_형태소 if isinstance(review_형태소, str) else ''
    grams = build_gram_set(text)
    return {attr: int(any(kw in grams for kw in kws)) for attr, kws in ATTRIBUTE_KEYWORDS_NORM.items()}

## 4. 데이터셋 생성

국가별 데이터를 합치면서, 리뷰마다 새 `리뷰_번호`를 만들고 속성 9개를 매칭.

In [8]:
model_rows = []
reference_rows = []

for key, df in country_dfs.items():
    info = COUNTRY_INFO[key]
    code_, name = info['code'], info['name']
    for _, row in df.iterrows():
        review_id = f"{code_}_P{int(row['제품_순위']):02d}_{int(row['리뷰_번호']):04d}"
        형태소 = row['리뷰_형태소'] if isinstance(row['리뷰_형태소'], str) else ''
        attrs = match_attributes(형태소)

        model_rows.append({
            '리뷰_번호': review_id,
            '국가': name,
            '제품_순위': row['제품_순위'],
            '별점': row['별점'],
            'Y': row['Y'],
            **attrs,
        })
        reference_rows.append({
            '리뷰_번호': review_id,
            '국가': name,
            '제품_순위': row['제품_순위'],
            '별점': row['별점'],
            'Y': row['Y'],
            '리뷰_원문': row['리뷰_원문'],
            '리뷰_불용어처리': row['리뷰_불용어처리'],
            '리뷰_형태소': 형태소,
        })

model_df = pd.DataFrame(model_rows)
reference_df = pd.DataFrame(reference_rows)

print('토픽분류 데이터셋:', model_df.shape)
print('리뷰_번호 중복 개수:', model_df['리뷰_번호'].duplicated().sum())
model_df.head()

토픽분류 데이터셋: (7230, 14)
리뷰_번호 중복 개수: 0


,리뷰_번호,국가,제품_순위,별점,Y,보습,향,발림성,자극,용기,배송,기능,가격,브랜드 충성도
0,RU_P01_0001,러시아,1,5,1,0,0,1,0,1,0,1,0,0
1,RU_P01_0002,러시아,1,5,1,1,0,0,0,0,0,0,0,0
2,RU_P01_0003,러시아,1,5,1,1,1,0,0,1,0,1,0,0
3,RU_P01_0004,러시아,1,5,1,0,0,0,0,0,0,0,0,0
4,RU_P01_0005,러시아,1,5,1,0,1,1,0,1,1,0,0,0


## 5. 결과 확인

In [9]:
print('국가별 리뷰 수')
print(model_df['국가'].value_counts())
print()
print('속성별 1의 비율 (전체)')
print(model_df[ATTRIBUTE_COLUMNS].mean().round(3))
print()
print('Y 분포')
print(model_df['Y'].value_counts())
model_df.head(10)

국가별 리뷰 수
국가
한국     1939
미국     1907
러시아    1842
일본     1542
Name: count, dtype: int64

속성별 1의 비율 (전체)
보습         0.429
향          0.152
발림성        0.436
자극         0.421
용기         0.193
배송         0.059
기능         0.342
가격         0.076
브랜드 충성도    0.100
dtype: float64

Y 분포
Y
1    4815
0    2415
Name: count, dtype: int64


,리뷰_번호,국가,제품_순위,별점,Y,보습,향,발림성,자극,용기,배송,기능,가격,브랜드 충성도
0,RU_P01_0001,러시아,1,5,1,0,0,1,0,1,0,1,0,0
1,RU_P01_0002,러시아,1,5,1,1,0,0,0,0,0,0,0,0
2,RU_P01_0003,러시아,1,5,1,1,1,0,0,1,0,1,0,0
3,RU_P01_0004,러시아,1,5,1,0,0,0,0,0,0,0,0,0
4,RU_P01_0005,러시아,1,5,1,0,1,1,0,1,1,0,0,0
5,RU_P01_0006,러시아,1,5,1,1,0,0,0,0,0,0,0,0
6,RU_P01_0007,러시아,1,5,1,0,0,0,0,1,0,0,0,0
7,RU_P01_0008,러시아,1,5,1,0,1,1,0,0,0,0,1,0
8,RU_P01_0009,러시아,1,5,1,0,0,0,0,0,0,0,0,0
9,RU_P01_0010,러시아,1,5,1,0,1,1,0,0,0,1,0,0


## 6. 엑셀로 저장

토픽분류 파일과, 리뷰 원문/번역을 보관하는 해석용 참고 파일을 따로 저장.

In [10]:
MODEL_FILE = '토픽분류_데이터셋.xlsx'
REFERENCE_FILE = '토픽분류_데이터셋_해석용_참고파일.xlsx'

model_df.to_excel(MODEL_FILE, index=False)
reference_df.to_excel(REFERENCE_FILE, index=False)

print(f'저장 완료: {MODEL_FILE}, {REFERENCE_FILE}')

from google.colab import files
files.download(MODEL_FILE)
files.download(REFERENCE_FILE)

저장 완료: 토픽분류_데이터셋.xlsx, 토픽분류_데이터셋_해석용_참고파일.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>